# 7-2. 지시 튜닝 — 대화 한 건이 모델 입력이 되기까지

지시 튜닝(instruction tuning / SFT)이 하는 일은 한 줄입니다.

> **사전훈련에서 배운 것을, 대답의 형식에 맞춰 끌어낸다.**

새 지식을 넣는 것이 아닙니다. 끌어낼 것이 없으면 아무것도 나오지 않습니다.
지시 데이터가 사전훈련 데이터보다 훨씬 적어도 되는 이유도 같습니다 —
**언어와 지식은 이미 배웠고, 여기서는 꺼내는 방법만** 배웁니다.

이 노트북에서 직접 만들 것:

1. 대화 → ChatML 문자열
2. ChatML → `input_ids` / `labels` (**손실 마스킹**)
3. 마스킹 유무가 학습 결과를 어떻게 바꾸는지 실제 훈련으로 확인
4. 긴 대화 쪼개기, 배치 패딩
5. 공개 한국어 지시 데이터셋 둘러보기

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))
sys.path.insert(0, os.path.abspath(".."))

import torch, torch.nn.functional as F
from common import set_seed, get_device
from chat import (render_chatml, build_sft_example, preview_mask, collate,
                  split_long_conversation, load_tokenizer,
                  IGNORE_INDEX, IM_START, IM_END)

set_seed(1234)
device = get_device()

# transformers 가 설치돼 있으면 Qwen3 토크나이저를, 없으면 바이트 토크나이저를 씁니다.
tok = load_tokenizer("Qwen/Qwen3-0.6B-Base")
pad_id = getattr(tok, "pad_token_id", None) or 0

## 1. 대화 템플릿 (ChatML)

메시지 하나를 감싸는 방식은 **역할에 상관없이 같습니다.**

```
<|im_start|>역할
내용<|im_end|>
```

시작 토큰 다음에 역할 이름, 줄 바꿔 내용, 종료 토큰으로 닫습니다.
`<|im_start|>` `<|im_end|>` 는 **글자가 아니라 토큰 하나**입니다 —
모델이 「여기서 차례가 바뀐다」를 글자 패턴이 아니라 토큰 하나로 배우게 하려는 것입니다.

In [ ]:
sample = [
    {"role": "system",    "content": "당신은 항상 설명을 제공하는 도움이 되는 비서입니다. 5세 아이에게 대답한다고 생각하세요."},
    {"role": "user",      "content": "현 유엔 사무총장은 누구인가요?"},
    {"role": "assistant", "content": "현재 유엔 사무총장은 안토니오 구테헤스라는 사람입니다. 세계 여러 나라가 모인 큰 모임의 대표예요."},
]

print(render_chatml(sample))
print("=" * 60)
print("대화할 때는 여기까지만 넣고 모델이 그 뒤를 씁니다:")
print(repr(render_chatml(sample[:2], add_generation_prompt=True)))

## 2. 손실 마스킹 — 읽기는 하고, 배우지는 않는다

같은 데이터를 두 방식으로 놓고 보겠습니다.
**░ 로 가린 부분이 「읽기만 하고 손실은 계산하지 않는」 토큰**입니다.

In [ ]:
for title, mask in [("마스킹 없음 · 데이터 전체에 손실", False),
                    ("마스킹 있음 · assistant 응답 + 종료 토큰에만 손실", True)]:
    ex = build_sft_example(tok, sample, mask_prompt=mask)
    print(f"── {title}")
    print(preview_mask(tok, ex))
    print(f"   전체 {len(ex)} 토큰 중 손실 대상 {ex.n_supervised} 토큰 "
          f"({ex.n_supervised/len(ex):.0%})\n")

### 두 가지 세부 결정

**(1) `<|im_start|>assistant` 머리까지 가린다**
대화할 때 이 부분은 **우리가 붙여 줍니다.** 모델은 그 뒤부터 씁니다.
가리지 않으면 모델이 자기 차례를 스스로 열고 들어오는 법까지 배웁니다.

**(2) 종료 토큰은 가리지 않는다**
멈춤은 별도의 장치가 아닙니다. 모델은 언제나 다음 토큰을 고를 뿐이고,
「멈추는 법을 배운다」는 것은 **답이 끝나는 그 지점에서 `<|im_end|>` 를 고르는 법**을
배운다는 뜻입니다. 그래서 종료 토큰은 반드시 손실에 넣습니다.

In [ ]:
ex = build_sft_example(tok, sample, mask_prompt=True)
sup_ids = [i for i, l in zip(ex.input_ids, ex.labels) if l != IGNORE_INDEX]

print("손실이 걸리는 구간의 맨 앞 :", repr(tok.decode(sup_ids[:12])))
print("손실이 걸리는 구간의 맨 뒤 :", repr(tok.decode(sup_ids[-6:])))
eos_id = tok.encode(IM_END, add_special_tokens=False)[0]
print(f"\n종료 토큰(id={eos_id})이 손실 대상에 들어 있는가? -> {eos_id in sup_ids}")

head_ids = tok.encode(f"{IM_START}assistant\n", add_special_tokens=False)
print(f"'<|im_start|>assistant' 머리가 손실 대상에서 빠졌는가? -> "
      f"{not any(h in sup_ids[:len(head_ids)] for h in head_ids[:1])}")

## 3. 마스킹이 정말 결과를 바꾸는가

말로 넘어가지 말고 돌려 봅니다. 아주 작은 트랜스포머를 **같은 데이터로 두 번** 훈련합니다.
차이는 `mask_prompt` 뿐입니다. 훈련 후 **질문 없이 빈 상태에서 생성**시켜 보면,
마스킹 없이 배운 쪽은 **자기가 질문까지 지어내기 시작합니다.**

In [ ]:
from chat import ByteTokenizer
btok = ByteTokenizer()          # 작은 어휘로 빠르게 돌리기 위해 바이트 토크나이저 사용

mini_data = [
    [{"role": "user", "content": "안녕?"},        {"role": "assistant", "content": "안녕하세요!"}],
    [{"role": "user", "content": "이름이 뭐야?"},  {"role": "assistant", "content": "저는 비서입니다."}],
    [{"role": "user", "content": "고마워"},        {"role": "assistant", "content": "천만에요."}],
    [{"role": "user", "content": "잘 자"},         {"role": "assistant", "content": "좋은 밤 되세요."}],
]

class TinyLM(torch.nn.Module):
    def __init__(self, vocab, d=96, nhead=4, nlayer=2, maxlen=256):
        super().__init__()
        self.emb = torch.nn.Embedding(vocab, d)
        self.pos = torch.nn.Embedding(maxlen, d)
        layer = torch.nn.TransformerEncoderLayer(d, nhead, d * 4, batch_first=True,
                                                 norm_first=True, dropout=0.0)
        self.body = torch.nn.TransformerEncoder(layer, nlayer)
        self.head = torch.nn.Linear(d, vocab)
    def forward(self, ids):
        T = ids.size(1)
        h = self.emb(ids) + self.pos(torch.arange(T, device=ids.device))
        m = torch.triu(torch.ones(T, T, dtype=torch.bool, device=ids.device), 1)
        return self.head(self.body(h, mask=m))

def train(mask_prompt, steps=600):
    set_seed(7)
    model = TinyLM(btok.vocab_size).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    exs = [build_sft_example(btok, d, mask_prompt=mask_prompt) for d in mini_data]
    batch = collate(exs, btok.pad_token_id)
    ids, labs = batch["input_ids"].to(device), batch["labels"].to(device)
    for s in range(steps):
        logits = model(ids)
        loss = F.cross_entropy(logits[:, :-1].reshape(-1, btok.vocab_size),
                               labs[:, 1:].reshape(-1), ignore_index=IGNORE_INDEX)
        opt.zero_grad(); loss.backward(); opt.step()
    return model, loss.item()

m_masked,   l1 = train(True)
m_unmasked, l2 = train(False)
print(f"최종 손실  마스킹 O: {l1:.4f}   마스킹 X: {l2:.4f}")

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new=60, greedy=True, temperature=1.0):
    """프롬프트는 빼고 **새로 생성한 부분만** 돌려줍니다."""
    ids = torch.tensor([btok.encode(prompt)], device=device)
    start = ids.size(1)
    for _ in range(max_new):
        logits = model(ids)[:, -1]
        nxt = (logits.argmax(-1, keepdim=True) if greedy else
               torch.multinomial(F.softmax(logits / temperature, -1), 1))
        ids = torch.cat([ids, nxt], 1)
        if nxt.item() == btok.eos_token_id:
            break
    new_ids = ids[0, start:].tolist()
    return btok.decode(new_ids), new_ids

# 대화 시작 토큰만 주고, 모델이 어떤 역할로 차례를 여는지 본다.
def opens_user_turn(model, trials=8):
    hits = 0
    for _ in range(trials):
        text, _ = generate(model, IM_START, max_new=12, greedy=False, temperature=0.8)
        if text.startswith("user"):
            hits += 1
    return hits, trials

set_seed(0); a = opens_user_turn(m_masked)
set_seed(0); b = opens_user_turn(m_unmasked)
print(f"마스킹 O 모델이 'user' 차례를 스스로 연 횟수 : {a[0]}/{a[1]}")
print(f"마스킹 X 모델이 'user' 차례를 스스로 연 횟수 : {b[0]}/{b[1]}")
print()
print("[마스킹 X 모델이 쓴 것]", repr(generate(m_unmasked, IM_START, max_new=40)[0][:120]))
print("[마스킹 O 모델이 쓴 것]", repr(generate(m_masked,   IM_START, max_new=40)[0][:120]))

> 마스킹 없이 배운 모델은 **user 차례를 스스로 엽니다.** 질문 쓰는 법까지 배웠기 때문입니다.
> 사람과 번갈아 말해야 하는 대화 모델에서는 이게 그대로 버그가 됩니다.
>
> 마스킹한 모델의 출력이 엉망으로 보이는 것은 정상입니다 — 차례를 **여는 법 자체를 배우지 않았으니**
> 아무 데서나 시작하면 할 말이 없습니다. 질문을 제대로 주면 멀쩡히 답합니다(아래 4절).
>
> (장난감 크기라 매번 깔끔하진 않습니다. 셀을 여러 번 돌려 경향을 보세요.)

## 4. 종료 토큰을 손실에서 빼면

「멈추는 법」을 배울 기회가 없어서 **답을 다 쓰고도 계속 이어 쓰다가
토큰 상한에서 잘립니다.** 훈련 전 베이스 모델이 보여주는 바로 그 모습입니다.

In [ ]:
def train_no_eos_loss(steps=600):
    set_seed(7)
    model = TinyLM(btok.vocab_size).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    exs = []
    for d in mini_data:
        e = build_sft_example(btok, d, mask_prompt=True)
        eid = btok.eos_token_id
        # 종료 토큰만 손실에서 뺀다 (나머지 마스킹은 그대로)
        e.labels = [IGNORE_INDEX if t == eid else l for t, l in zip(e.input_ids, e.labels)]
        exs.append(e)
    batch = collate(exs, btok.pad_token_id)
    ids, labs = batch["input_ids"].to(device), batch["labels"].to(device)
    for _ in range(steps):
        logits = model(ids)
        loss = F.cross_entropy(logits[:, :-1].reshape(-1, btok.vocab_size),
                               labs[:, 1:].reshape(-1), ignore_index=IGNORE_INDEX)
        opt.zero_grad(); loss.backward(); opt.step()
    return model

m_noeos = train_no_eos_loss()
prompt = render_chatml([{"role": "user", "content": "안녕?"}], add_generation_prompt=True)
LIMIT = 80

for name, model in [("종료 토큰 학습 O", m_masked), ("종료 토큰 학습 X", m_noeos)]:
    text, new_ids = generate(model, prompt, max_new=LIMIT)
    stopped = btok.eos_token_id in new_ids          # 프롬프트가 아니라 **생성분**에서 확인
    why = "스스로 멈춤" if stopped else f"상한({LIMIT})에서 잘림"
    print(f"{name} : 새로 쓴 토큰 {len(new_ids):3d}개 — {why}")
    print(f"   -> {text[:110]!r}\n")

## 5. 긴 대화와 배치 — 실무에서 반드시 밟는 두 단계

**(1) 컨텍스트 상한을 넘는 대화**는 길이에서 그냥 자르면 안 됩니다.
답이 중간에 끊긴 채로 「여기서 멈춰라」를 가르치게 됩니다.
**문답 쌍이 완결된 경계**에서 나눕니다.

**(2) 배치로 묶을 때 패딩 자리의 라벨은 `IGNORE_INDEX`** 입니다.
0으로 두면 0번 토큰을 맞히라고 가르치는 셈입니다.

In [ ]:
long_chat = [{"role": "system", "content": "짧게 답하세요."}]
for i in range(8):
    long_chat += [{"role": "user", "content": f"{i}번째 질문입니다. 조금 길게 적어 봅니다."},
                  {"role": "assistant", "content": f"{i}번째 답입니다."}]

chunks = split_long_conversation(long_chat, btok, max_len=200)
print(f"원본 {len(btok.encode(render_chatml(long_chat)))} 토큰 -> {len(chunks)} 조각")
for i, c in enumerate(chunks):
    n = len(btok.encode(render_chatml(c)))
    roles = "".join(m["role"][0] for m in c)
    print(f"  조각 {i}: {n:4d} 토큰  역할열 = {roles}   (완결된 쌍으로 끝나는가: {c[-1]['role']=='assistant'})")

In [ ]:
# 길이가 제각각인 샘플들을 한 배치로 묶어 봅니다.
varied = [build_sft_example(btok, d, mask_prompt=True) for d in mini_data]
batch = collate(varied, btok.pad_token_id)

print("샘플별 길이:", [len(e) for e in varied], "-> 배치", tuple(batch["input_ids"].shape))
for row in range(len(varied)):
    labs, attn = batch["labels"][row], batch["attention_mask"][row]
    n_real = attn.sum().item(); n_pad = len(attn) - n_real
    ok_lab  = bool((labs[n_real:] == IGNORE_INDEX).all())
    ok_attn = bool((attn[n_real:] == 0).all())
    print(f"  샘플 {row}: 실제 {n_real:3d} + 패딩 {n_pad:3d}   "
          f"패딩 라벨=IGNORE {ok_lab}   패딩 attn=0 {ok_attn}")

print("\n패딩 자리를 0 으로 두면 '0번 토큰을 맞혀라'를 가르치는 셈이 됩니다.")

## 6. 공개 한국어 지시 데이터셋

강의가 실물로 보여준 데이터셋들입니다. 성격이 서로 달라서,
**하나만 고르지 말고 섞는** 것이 요령입니다.

| 데이터셋 | 성격 |
|---|---|
| [kyujinpy/OpenOrca-KO](https://huggingface.co/datasets/kyujinpy/OpenOrca-KO) | 시스템·사용자·어시스턴트 3역할 실물 |
| [beomi/KoAlpaca-v1.1a](https://huggingface.co/datasets/beomi/KoAlpaca-v1.1a) | 단일턴 (지식iN 질문 말투) |
| [nayohan/141_korean_multi_session_dialogue](https://huggingface.co/datasets/nayohan/141_korean_multi_session_dialogue) | 멀티턴 (사람–사람 대화) |
| [heegyu/korquad-chat-v1](https://huggingface.co/datasets/heegyu/korquad-chat-v1) | 위키 지문을 시스템에 넣은 지식 기반 대화 |
| [maywell/ko_wikidata_QA](https://huggingface.co/datasets/maywell/ko_wikidata_QA) | 지식 질문·답 |
| [MarkrAI/KoCommercial-Dataset](https://huggingface.co/datasets/MarkrAI/KoCommercial-Dataset) | 여러 종류의 일 |
| [changpt/ko-lima-vicuna](https://huggingface.co/datasets/changpt/ko-lima-vicuna) | 긴 답 · 품질 기준 |
| [jojo0217/korean_safe_conversation](https://huggingface.co/datasets/jojo0217/korean_safe_conversation) | 공감 응답 |
| [FreedomIntelligence/sharegpt-korean](https://huggingface.co/datasets/FreedomIntelligence/sharegpt-korean) | 긴 대화 |
| [nlpai-lab/openassistant-guanaco-ko](https://huggingface.co/datasets/nlpai-lab/openassistant-guanaco-ko) | 번역이 깨뜨린 예 (주의) |
| [nlpai-lab/kullm-v2](https://huggingface.co/datasets/nlpai-lab/kullm-v2) | 챗봇 말버릇이 밴 예 (강의에서는 제외) |

### 고를 때 조심할 것

- **가장 큰 것 하나를 고르는 것이 가장 위험합니다.** 큰 것은 대부분
  기계번역이거나 출처를 알 수 없는 혼합입니다.
- **모델이 만든 데이터**는 그 모델의 편향과 오류를 그대로 물려받습니다.
- **단일턴만 학습하면** 이어지는 질문에서 앞의 흐름을 놓칩니다. 멀티턴을 꼭 섞으세요.
- LIMA(2023)는 **잘 고른 천 건** 으로도 충분하다는 것을 보였습니다. 양보다 품질입니다.

In [ ]:
# datasets 가 설치돼 있을 때만 실제로 받아 봅니다 (수백 MB 받지 않도록 streaming).
try:
    from datasets import load_dataset
    ds = load_dataset("beomi/KoAlpaca-v1.1a", split="train", streaming=True)
    for i, row in enumerate(ds):
        if i >= 2: break
        msgs = [{"role": "user", "content": row["instruction"]},
                {"role": "assistant", "content": row["output"]}]
        ex = build_sft_example(tok, msgs)
        print(f"[{i}] {len(ex)} 토큰 / 손실 대상 {ex.n_supervised}")
        print("   Q:", row["instruction"][:60])
        print("   A:", row["output"][:60], "...\n")
except Exception as e:
    print(f"datasets 미설치 또는 네트워크 없음 ({type(e).__name__}) — 건너뜁니다.")
    print("설치:  ../../.venv311/bin/pip install -r ../requirements.txt")

## 정리

- 마스킹은 **labels 에만** 한다. input_ids 는 그대로 둔다.
- `<|im_start|>assistant` **머리는 가리고**, 내용과 **종료 토큰은 가리지 않는다.**
- 긴 대화는 **완결된 문답 경계**에서 나눈다.
- 패딩 라벨은 **IGNORE_INDEX**.
- 데이터는 **섞는다.** 하나만 크게 쓰지 않는다.

### 확인 문제
1. 가린 토큰을 입력에서도 빼면 무슨 일이 생깁니까?
2. 종료 토큰을 손실에 넣는 이유를 한 문장으로.
3. 단일턴 데이터만으로 학습한 모델의 약점은?

> **다음** → `0703_FullFineTuning.ipynb`
> 실제 베이스 모델에 이 파이프라인을 붙여 **전체 미세 조정**을 돌립니다.